In [ ]:
%pip install -q vllm sentence-transformers tqdm implicit

In [ ]:
import os

os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'

import gc
import json
import re
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import warnings
warnings.filterwarnings('ignore')

# ===== Константы =====
DATA_DIR = Path('../../data/children_products')
CLEANED_CSV = DATA_DIR / 'clildren_product_cleaned.csv'
RAW_CSV = DATA_DIR / '!03&04_17_VSE.csv'

MIN_INTERACTIONS = 5
TRAIN_SPLIT = 0.8

N_EVAL_USERS = 1000 
FINAL_K = 20
MAX_HISTORY_ITEMS = 30

MODEL_CHOICE = 'qwen7b'  # 'qwen3b' | 'qwen7b' | 'mistral7b' | 'vikhr12b' | 'qwen14b' | 'qwen32b_awq'
MODEL_REGISTRY = {
    'qwen3b':      ('Qwen/Qwen2.5-3B-Instruct',                          'qwen3b',      None,  6,  'qwen3b'),
    'qwen7b':      ('Qwen/Qwen2.5-7B-Instruct',                          'qwen7b',      None,  14, 'qwen7b'),
    'mistral7b':   ('mistralai/Mistral-7B-Instruct-v0.3',                'mistral7b',   None,  14, 'mistral7b'),
    'vikhr12b':    ('Vikhrmodels/Vikhr-Nemo-12B-Instruct-R-21-09-24',    'vikhr12b',    None,  24, 'vikhr12b'),
    'qwen14b':     ('Qwen/Qwen2.5-14B-Instruct',                         'qwen14b',     None,  28, 'qwen14b'),
    'qwen32b_awq': ('Qwen/Qwen2.5-32B-Instruct-AWQ',                     'qwen32b_awq', 'awq', 17, 'qwen32b_awq'),
}
LLM_MODEL, MODEL_SUFFIX, LLM_QUANT, _vram_est, _note = MODEL_REGISTRY[MODEL_CHOICE]
print(f'Using LLM: {LLM_MODEL}  (~{_vram_est} GB, {_note})')

LLM_ZS_V2_CACHE = DATA_DIR / f'llm_zero_shot_v2_cache_vllm_{MODEL_SUFFIX}.json'

RRF_K = 60
ALPHA_SWEEP = [0.0, 0.1, 0.3, 0.5, 1.0]

PRICE_TIER_QUANTILES = (0.33, 0.66)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [ ]:
df = pd.read_csv(CLEANED_CSV)
print(f'Исходный датасет: {df.shape}')

raw = pd.read_csv(
    RAW_CSV,
    sep=';', encoding='cp1251',
    usecols=['ID_SKU', 'Номенклатура', 'Группа2', 'Группа3', 'Тип'],
    dtype=str, low_memory=False
)
raw = raw.dropna(subset=['ID_SKU']).drop_duplicates('ID_SKU')

df = df.merge(raw[['ID_SKU', 'Номенклатура']], on='ID_SKU', how='left')
fallback = df['Группа3'].fillna('') + ' / ' + df['Тип'].fillna('')
df['Номенклатура'] = df['Номенклатура'].fillna(fallback)

df_filtered = df[(df['Статус'] == 'Доставлен') & (df['Отменено'] == 'Нет')].copy()
df_filtered = df_filtered.dropna(subset=['Телефон_new', 'ID_SKU', 'Дата'])
df_filtered['Дата'] = pd.to_datetime(df_filtered['Дата'], errors='coerce')
df_filtered = df_filtered.dropna(subset=['Дата'])

for _ in range(5):
    uc = df_filtered.groupby('Телефон_new').size()
    ic = df_filtered.groupby('ID_SKU').size()
    active_users = uc[uc >= MIN_INTERACTIONS].index
    active_items = ic[ic >= MIN_INTERACTIONS].index
    before = len(df_filtered)
    df_filtered = df_filtered[
        df_filtered['Телефон_new'].isin(active_users) &
        df_filtered['ID_SKU'].isin(active_items)
    ]
    if len(df_filtered) == before:
        break

user_enc, item_enc = LabelEncoder(), LabelEncoder()
df_filtered['user_id'] = user_enc.fit_transform(df_filtered['Телефон_new'])
df_filtered['item_id'] = item_enc.fit_transform(df_filtered['ID_SKU'])
n_users, n_items = len(user_enc.classes_), len(item_enc.classes_)
print(f'Пользователей: {n_users:,}  Товаров: {n_items:,}  Взаимодействий: {len(df_filtered):,}')

df_filtered = df_filtered.sort_values('Дата')
split_date = df_filtered['Дата'].quantile(TRAIN_SPLIT)
train_data = df_filtered[df_filtered['Дата'] < split_date].copy()
test_data = df_filtered[df_filtered['Дата'] >= split_date].copy()
print(f'Split date: {split_date}')

train_int = train_data.groupby(['user_id', 'item_id']).size().reset_index(name='count')
test_int = test_data.groupby(['user_id', 'item_id']).size().reset_index(name='count')
train_matrix = csr_matrix(
    (train_int['count'].values, (train_int['user_id'].values, train_int['item_id'].values)),
    shape=(n_users, n_items)
)

test_user_items = test_int.groupby('user_id')['item_id'].apply(list).to_dict()
train_user_set = set(train_int['user_id'].unique())
warm_eval_users = [u for u in test_user_items if u in train_user_set]
all_test_users = list(test_user_items.keys())
cold_eval_users = [u for u in all_test_users if u not in train_user_set]

rng = np.random.default_rng(RANDOM_SEED)
sample_users = rng.choice(warm_eval_users, size=min(N_EVAL_USERS, len(warm_eval_users)), replace=False)
sample_users = sorted(map(int, sample_users))
print(f'Warm users: {len(warm_eval_users)}, cold users: {len(cold_eval_users)}')
print(f'Подвыборка для оценки: {len(sample_users)} пользователей (warm)')


In [ ]:
# Метаданные товара
item_features = train_data.groupby('item_id').agg(
    item_avg_price=('Цена', 'mean'),
    item_n_purchases=('item_id', 'size'),
).reset_index()

for col in ['Группа2', 'Группа3', 'Тип']:
    if col in train_data.columns:
        modal = train_data.groupby('item_id')[col].agg(
            lambda x: x.mode().iloc[0] if len(x.mode()) else 'Неизвестно'
        ).reset_index()
        modal.columns = ['item_id', col]
        item_features = item_features.merge(modal, on='item_id', how='left')

item_meta = item_features.set_index('item_id').to_dict('index')
print(f'Items с метаданными: {len(item_meta)}')

# Ценовые тиры
prices = item_features['item_avg_price'].dropna().values
q_low, q_high = np.quantile(prices, PRICE_TIER_QUANTILES)
print(f'Price tier thresholds: low<{q_low:.0f} ≤ mid <{q_high:.0f} ≤ high')


def price_to_tier(p: float) -> str:
    if p is None or np.isnan(p):
        return 'mid'
    if p < q_low:
        return 'low'
    if p < q_high:
        return 'mid'
    return 'high'


item_features['price_tier'] = item_features['item_avg_price'].apply(price_to_tier)
item_meta = item_features.set_index('item_id').to_dict('index')

# Каталожные индексы (по убыванию популярности)
cat2_to_items = defaultdict(list)
cat23_to_items = defaultdict(list)
for iid, meta in sorted(item_meta.items(),
                        key=lambda kv: -kv[1].get('item_n_purchases', 0)):
    g2 = meta.get('Группа2', 'Неизвестно')
    g3 = meta.get('Группа3', 'Неизвестно')
    cat2_to_items[g2].append(int(iid))
    cat23_to_items[(g2, g3)].append(int(iid))

available_g2 = sorted([g for g in cat2_to_items if g != 'Неизвестно'])
available_g3 = sorted({g3 for (_, g3) in cat23_to_items if g3 != 'Неизвестно'})
print(f'Группа2: {len(available_g2)} категорий, Группа3: {len(available_g3)}')
print(f'Топ Группа2 по популярности: {sorted(cat2_to_items, key=lambda g: -len(cat2_to_items[g]))[:5]}')


In [ ]:
train_sorted = train_data.sort_values('Дата')
user_history_cache = {}
for uid, g in train_sorted.groupby('user_id'):
    rows = []
    for _, r in g.iterrows():
        date = r['Дата'].strftime('%Y-%m-%d') if pd.notna(r['Дата']) else ''
        rows.append({
            'date': date,
            'name': r['Номенклатура'],
            'g2': r.get('Группа2', '?'),
            'g3': r.get('Группа3', '?'),
            'price': r.get('Цена', 0) or 0,
        })
    user_history_cache[int(uid)] = rows

user_train_items_cache = {
    int(uid): set(g['item_id'].tolist())
    for uid, g in train_sorted.groupby('user_id')
}

print(f'История кэширована для {len(user_history_cache)} юзеров')


def build_user_history(user_id: int, max_items: int = MAX_HISTORY_ITEMS) -> str:
    items = user_history_cache.get(user_id, [])
    tail = items[-max_items:]
    return '\n'.join(
        f'- {r["date"]}: {r["name"]} | {r["g2"]} > {r["g3"]} | {r["price"]:.0f}р'
        for r in tail
    )


print('\nПример истории uid={}:'.format(sample_users[0]))
print(build_user_history(sample_users[0], 5))


## Загружаем vLLM

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

_eot_id = tokenizer.convert_tokens_to_ids('<|eot_id|>')
LLM_TERMINATORS = sorted({
    int(t) for t in [tokenizer.eos_token_id, _eot_id]
    if isinstance(t, int) and t is not None and t >= 0
})

_llm_kwargs = dict(
    model=LLM_MODEL,
    max_model_len=4096,
    gpu_memory_utilization=0.9,
    enforce_eager=False,
    trust_remote_code=False,
)
if LLM_QUANT is None:
    _llm_kwargs['dtype'] = 'bfloat16'
else:
    _llm_kwargs['quantization'] = LLM_QUANT
    _llm_kwargs['dtype'] = 'auto'
llm = LLM(**_llm_kwargs)
print('vLLM loaded')


In [ ]:
ALLOWED_G2_STR = ', '.join(available_g2)

SYSTEM_PROMPT_ZS_V2 = (
    'Ты — рекомендательная система детского интернет-магазина. '
    'На основе истории покупок проанализируй пользователя и заполни JSON. '
    'НЕ предлагай конкретные товары — только характеристики предпочтений. '
    'Категории Группа2 выбирай ИСКЛЮЧИТЕЛЬНО из списка: ' + ALLOWED_G2_STR + '. '
    'Группа3 — короткие названия подкатегорий (как в истории), 5–10 элементов. '
    'price_tier ∈ {low, mid, high}. '
    'Отвечай СТРОГО в формате JSON без пояснений.'
)

JSON_SCHEMA_HINT = (
    '{\n'
    '  "predicted_age_range": "0-1 / 1-3 / 3-6 / 6-12 / 12+",\n'
    '  "top_categories_g2": ["...", "...", "..."],\n'
    '  "top_categories_g3": ["...", "...", ...],\n'
    '  "price_tier": "low / mid / high",\n'
    '  "interests_keywords": ["...", "...", ...]\n'
    '}'
)


def build_zs_v2_prompt(user_id: int) -> str:
    hist = build_user_history(user_id, max_items=MAX_HISTORY_ITEMS)
    user_prompt = (
        f'История покупок пользователя (последние {MAX_HISTORY_ITEMS}):\n{hist}\n\n'
        f'Заполни JSON по схеме:\n{JSON_SCHEMA_HINT}\n\n'
        f'Только JSON, без поясняющего текста.'
    )
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT_ZS_V2},
        {'role': 'user', 'content': user_prompt},
    ]
    return tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)


# Пример
demo_uid = sample_users[0]
print(build_zs_v2_prompt(demo_uid)[:1500])
print('...')


In [ ]:
def parse_zs_attributes(raw_text: str) -> dict:
    out = {
        'predicted_age_range': None,
        'top_categories_g2': [],
        'top_categories_g3': [],
        'price_tier': None,
        'interests_keywords': [],
    }
    obj = None
    try:
        obj = json.loads(raw_text)
    except Exception:
        m = re.search(r'\{[\s\S]*\}', raw_text)
        if m:
            try:
                obj = json.loads(m.group(0))
            except Exception:
                obj = None
    if not isinstance(obj, dict):
        return out
    out['predicted_age_range'] = obj.get('predicted_age_range') or None
    g2 = obj.get('top_categories_g2') or obj.get('top_g2') or []
    if isinstance(g2, list):
        out['top_categories_g2'] = [str(x) for x in g2 if str(x).strip()]
    g3 = obj.get('top_categories_g3') or obj.get('top_g3') or []
    if isinstance(g3, list):
        out['top_categories_g3'] = [str(x) for x in g3 if str(x).strip()]
    pt = obj.get('price_tier')
    if isinstance(pt, str) and pt.lower() in ('low', 'mid', 'high'):
        out['price_tier'] = pt.lower()
    kw = obj.get('interests_keywords') or obj.get('keywords') or []
    if isinstance(kw, list):
        out['interests_keywords'] = [str(x) for x in kw if str(x).strip()]
    return out


In [ ]:
zs_v2_cache = {'raw': {}, 'parsed': {}}
if LLM_ZS_V2_CACHE.exists():
    with open(LLM_ZS_V2_CACHE, 'r', encoding='utf-8') as f:
        loaded = json.load(f)
    zs_v2_cache['raw'] = {int(k): v for k, v in loaded.get('raw', {}).items()}
    zs_v2_cache['parsed'] = {int(k): v for k, v in loaded.get('parsed', {}).items()}
    print(f'Cache v2: восстановлено {len(zs_v2_cache["raw"])} ответов')

to_process = [uid for uid in sample_users if int(uid) not in zs_v2_cache['raw']]
print(f'К обработке: {len(to_process)} пользователей')

if to_process:
    prompts = [build_zs_v2_prompt(uid) for uid in to_process]
    sample_lens = [len(tokenizer.encode(p)) for p in prompts[:30]]
    p50, p95 = np.percentile(sample_lens, [50, 95])
    print(f'Длина промпта: p50={p50:.0f}, p95={p95:.0f} токенов')

    sampling_params = SamplingParams(
        temperature=0.0,
        max_tokens=400,
        stop_token_ids=LLM_TERMINATORS,
    )

    t0 = time.time()
    outputs = llm.generate(prompts, sampling_params)
    elapsed = time.time() - t0
    print(f'Generated {len(prompts)} в {elapsed:.1f}s ({elapsed / len(prompts):.2f}s/юзер)')

    for uid, out in zip(to_process, outputs):
        text = out.outputs[0].text
        zs_v2_cache['raw'][int(uid)] = text
        zs_v2_cache['parsed'][int(uid)] = parse_zs_attributes(text)

    # Парсим всё ещё нераспарсенное
    for uid in sample_users:
        uid = int(uid)
        if uid not in zs_v2_cache['parsed'] and uid in zs_v2_cache['raw']:
            zs_v2_cache['parsed'][uid] = parse_zs_attributes(zs_v2_cache['raw'][uid])

    with open(LLM_ZS_V2_CACHE, 'w', encoding='utf-8') as f:
        json.dump(
            {'raw': {str(k): v for k, v in zs_v2_cache['raw'].items()},
             'parsed': {str(k): v for k, v in zs_v2_cache['parsed'].items()}},
            f, ensure_ascii=False
        )
    print(f'Кэш сохранён в {LLM_ZS_V2_CACHE}')

# Диагностика парсинга
n_with_g2 = sum(1 for uid in sample_users
                if zs_v2_cache['parsed'].get(int(uid), {}).get('top_categories_g2'))
n_with_age = sum(1 for uid in sample_users
                 if zs_v2_cache['parsed'].get(int(uid), {}).get('predicted_age_range'))
n_with_tier = sum(1 for uid in sample_users
                  if zs_v2_cache['parsed'].get(int(uid), {}).get('price_tier'))
print(f'Распарсено top_categories_g2 для {n_with_g2}/{len(sample_users)} юзеров')
print(f'Распарсен predicted_age_range для {n_with_age}/{len(sample_users)}')
print(f'Распарсен price_tier для {n_with_tier}/{len(sample_users)}')


In [ ]:
# Глобальная популярность
global_pop = sorted(
    item_meta.items(), key=lambda kv: -kv[1].get('item_n_purchases', 0)
)
global_top_items = [int(iid) for iid, _ in global_pop[:200]]
max_pop = max(meta.get('item_n_purchases', 0) for meta in item_meta.values()) or 1


def normalize_text(s: str) -> str:
    return (s or '').strip().upper()


# Префиксное соответствие: предсказанные g2/g3 могут быть с легкой переформулировкой
g2_norm_to_orig = {normalize_text(g): g for g in available_g2}
g3_norm_to_orig = {normalize_text(g): g for g in available_g3}


def resolve_g2(name: str) -> str:
    if not name:
        return None
    nn = normalize_text(name)
    if nn in g2_norm_to_orig:
        return g2_norm_to_orig[nn]
    # try contains
    for n_orig, orig in g2_norm_to_orig.items():
        if nn in n_orig or n_orig in nn:
            return orig
    return None


def resolve_g3(name: str) -> str:
    if not name:
        return None
    nn = normalize_text(name)
    if nn in g3_norm_to_orig:
        return g3_norm_to_orig[nn]
    for n_orig, orig in g3_norm_to_orig.items():
        if nn in n_orig or n_orig in nn:
            return orig
    return None


def attribute_retrieve(parsed: dict, user_train_items: set,
                       final_k: int = FINAL_K, pool_per_cat: int = 30) -> list:
    if not parsed:
        candidates = list(global_top_items)
    else:
        g2_resolved = [resolve_g2(g) for g in parsed.get('top_categories_g2', [])]
        g2_resolved = [g for g in g2_resolved if g is not None]
        g3_resolved = [resolve_g3(g) for g in parsed.get('top_categories_g3', [])]
        g3_resolved = [g for g in g3_resolved if g is not None]

        candidates = []
        seen = set()

        # Предпочитаем (g2, g3) пары
        if g3_resolved:
            for g2 in g2_resolved or available_g2:
                for g3 in g3_resolved:
                    key = (g2, g3)
                    for iid in cat23_to_items.get(key, [])[:pool_per_cat]:
                        if iid not in seen:
                            seen.add(iid)
                            candidates.append(iid)

        # Затем по g2
        for g2 in g2_resolved:
            for iid in cat2_to_items.get(g2, [])[:pool_per_cat]:
                if iid not in seen:
                    seen.add(iid)
                    candidates.append(iid)

        # Fallback — глобальные топы
        if not candidates:
            candidates = list(global_top_items)
        elif len(candidates) < final_k * 3:
            for iid in global_top_items:
                if iid not in seen:
                    seen.add(iid)
                    candidates.append(iid)
                    if len(candidates) >= final_k * 5:
                        break

    # Скоринг
    target_tier = (parsed or {}).get('price_tier')
    scores = {}
    for iid in candidates:
        if iid in user_train_items:
            continue
        meta = item_meta.get(iid, {})
        pop = meta.get('item_n_purchases', 0) / max_pop
        tier = meta.get('price_tier', 'mid')
        bonus = 0.3 if (target_tier and tier == target_tier) else 0.0
        scores[iid] = pop * (1 + bonus)

    ordered = sorted(scores.keys(), key=lambda x: -scores[x])
    return ordered[:final_k]


# Базовый ретрив для всех warm-юзеров
zs_v2_recs = {}
for uid in tqdm(sample_users, desc='zs v2 retrieve'):
    parsed = zs_v2_cache['parsed'].get(int(uid), {})
    user_bought = user_train_items_cache.get(int(uid), set())
    zs_v2_recs[int(uid)] = attribute_retrieve(parsed, user_bought)

n_full = sum(1 for v in zs_v2_recs.values() if len(v) >= FINAL_K)
n_short = sum(1 for v in zs_v2_recs.values() if 0 < len(v) < FINAL_K)
n_empty = sum(1 for v in zs_v2_recs.values() if not v)
print(f'Полный список ({FINAL_K}): {n_full}, короткий: {n_short}, пустой: {n_empty}')


In [ ]:
try:
    from implicit.gpu.als import AlternatingLeastSquares
except ImportError:
    from implicit.als import AlternatingLeastSquares

als = AlternatingLeastSquares(
    factors=50, regularization=0.01, iterations=15,
    calculate_training_loss=True, random_state=RANDOM_SEED
)
als.fit(train_matrix)

K_CANDIDATES = 50
als_candidates = {}
als_recs_top_final = {}
for uid in tqdm(sample_users, desc='ALS top-50'):
    ids, scores = als.recommend(
        uid, train_matrix[uid],
        N=K_CANDIDATES, filter_already_liked_items=True
    )
    als_candidates[int(uid)] = list(zip([int(x) for x in ids], [float(s) for s in scores]))
    als_recs_top_final[int(uid)] = [int(x) for x in ids[:FINAL_K]]

print('ALS recommend готов')


In [ ]:
def build_rank_dict(rec_list, fallback_rank: int):
    return {int(iid): i + 1 for i, iid in enumerate(rec_list)}, fallback_rank


def rrf_fuse(als_list, zs_list, alpha: float, final_k: int = FINAL_K, k_rrf: int = RRF_K):
    n_als, n_zs = len(als_list), len(zs_list)
    rank_als = {int(iid): i + 1 for i, iid in enumerate(als_list)}
    rank_zs = {int(iid): i + 1 for i, iid in enumerate(zs_list)}
    all_items = set(rank_als) | set(rank_zs)
    fb_als = (n_als + 1) if n_als else 1
    fb_zs = (n_zs + 1) if n_zs else 1
    scores = {
        iid: 1.0 / (k_rrf + rank_als.get(iid, fb_als))
             + alpha * 1.0 / (k_rrf + rank_zs.get(iid, fb_zs))
        for iid in all_items
    }
    ordered = sorted(all_items, key=lambda x: -scores[x])
    return ordered[:final_k]


hybrid_recs = {}
for alpha in ALPHA_SWEEP:
    hybrid_recs[alpha] = {
        int(uid): rrf_fuse(
            als_recs_top_final[int(uid)],
            zs_v2_recs[int(uid)],
            alpha=alpha,
        )
        for uid in sample_users
    }

# Sanity-check: alpha=0 должен совпасть с ALS top-FINAL_K
match = all(hybrid_recs[0.0][u] == als_recs_top_final[u] for u in sample_users)
print(f'Sanity (alpha=0 vs ALS top-{FINAL_K}): {"OK" if match else "FAIL"}')


In [ ]:
K_VALUES = [5, 10, 20]


def precision_at_k(recommended, relevant, k):
    rec_k = set(recommended[:k])
    return len(rec_k & set(relevant)) / len(rec_k) if rec_k else 0.0


def recall_at_k(recommended, relevant, k):
    rec_k = set(recommended[:k])
    return len(rec_k & set(relevant)) / len(set(relevant)) if relevant else 0.0


def map_at_k(recommended, relevant, k):
    relevant = set(relevant)
    if not relevant:
        return 0.0
    score, hits = 0.0, 0.0
    for i, item in enumerate(recommended[:k]):
        if item in relevant:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(relevant), k)


def ndcg_at_k(recommended, relevant, k):
    relevant = set(relevant)
    if not relevant:
        return 0.0
    dcg = sum(1.0 / np.log2(i + 2) for i, item in enumerate(recommended[:k]) if item in relevant)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0


def aggregate(recs_by_user, users):
    out = {k: {m: [] for m in ['precision', 'recall', 'map', 'ndcg']} for k in K_VALUES}
    for u in users:
        u = int(u)
        rel = test_user_items.get(u, [])
        if not rel:
            continue
        rec = recs_by_user.get(u, [])
        for k in K_VALUES:
            out[k]['precision'].append(precision_at_k(rec, rel, k))
            out[k]['recall'].append(recall_at_k(rec, rel, k))
            out[k]['map'].append(map_at_k(rec, rel, k))
            out[k]['ndcg'].append(ndcg_at_k(rec, rel, k))
    return {k: {m: float(np.mean(v)) if v else 0.0 for m, v in metrics.items()}
            for k, metrics in out.items()}


all_results = {}
all_results['ALS'] = aggregate(als_recs_top_final, sample_users)
all_results['LLM zero-shot v2 (attr+catalog)'] = aggregate(zs_v2_recs, sample_users)
for alpha in ALPHA_SWEEP:
    label = f'zs v2 + ALS (RRF α={alpha})'
    all_results[label] = aggregate(hybrid_recs[alpha], sample_users)

print(f'Конфигураций: {len(all_results)}')


In [ ]:
rows = []
for name, res in all_results.items():
    row = {'model': name}
    for k in K_VALUES:
        for m in ('precision', 'recall', 'map', 'ndcg'):
            row[f'{m}@{k}'] = res[k][m]
    rows.append(row)
comp_df = pd.DataFrame(rows).set_index('model')
print('Сводная таблица:')
display(comp_df.style.format('{:.4f}'))


In [ ]:
# Прирост гибридов vs ALS
delta_rows = []
base_res = all_results['ALS']
for name, res in all_results.items():
    if name == 'ALS':
        continue
    delta = {
        f'{m}@{k}': (res[k][m] - base_res[k][m]) / (base_res[k][m] + 1e-12) * 100
        for k in K_VALUES for m in ('precision', 'recall', 'map', 'ndcg')
    }
    delta_rows.append({'model': name, **delta})

delta_df = pd.DataFrame(delta_rows).set_index('model')
print('\nΔ относительно ALS (%):')
display(delta_df.style.format('{:+.1f}%'))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics_list = ['precision', 'recall', 'map', 'ndcg']

for ax, metric in zip(axes.flat, metrics_list):
    for name, res in all_results.items():
        marker = 'o' if name == 'ALS' else ('s' if 'v2' in name and 'RRF' not in name else '^')
        ls = '-' if name == 'ALS' else ('--' if 'RRF' not in name else ':')
        lw = 2.5 if name == 'ALS' else 1.5
        ax.plot(K_VALUES, [res[k][metric] for k in K_VALUES],
                marker=marker, linestyle=ls, linewidth=lw, label=name)
    ax.set_xlabel('K')
    ax.set_ylabel(f'{metric.upper()}@K')
    ax.set_title(f'{metric.upper()}@K')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Zero-shot v2 vs ALS, гибрид (n={len(sample_users)})', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
uid = int(sample_users[0])
print(f'=== uid={uid} ===\n')
print('История (последние 10):')
for r in user_history_cache.get(uid, [])[-10:]:
    print(f'  {r["date"]}: {r["name"][:60]} | {r["g2"]} > {r["g3"]} | {r["price"]:.0f}р')

print('\n--- LLM raw ---')
print(zs_v2_cache['raw'].get(uid, '')[:600])

print('\n--- Parsed ---')
print(zs_v2_cache['parsed'].get(uid, {}))

print('\n--- ZS v2 retrieval top-10 ---')
for iid in zs_v2_recs[uid][:10]:
    meta = item_meta.get(iid, {})
    print(f'  {iid}: {meta.get("Группа2", "?")} > {meta.get("Группа3", "?")} | {meta.get("item_avg_price", 0):.0f}р')

print('\n--- ALS top-10 ---')
for iid in als_recs_top_final[uid][:10]:
    meta = item_meta.get(iid, {})
    print(f'  {iid}: {meta.get("Группа2", "?")} > {meta.get("Группа3", "?")} | {meta.get("item_avg_price", 0):.0f}р')

print('\n--- Hybrid α=0.3 top-10 ---')
for iid in hybrid_recs[0.3][uid][:10]:
    meta = item_meta.get(iid, {})
    print(f'  {iid}: {meta.get("Группа2", "?")} > {meta.get("Группа3", "?")} | {meta.get("item_avg_price", 0):.0f}р')

print('\n--- Test items ---')
for iid in test_user_items.get(uid, [])[:10]:
    meta = item_meta.get(iid, {})
    print(f'  {iid}: {meta.get("Группа2", "?")} > {meta.get("Группа3", "?")} | {meta.get("item_avg_price", 0):.0f}р')
